# B3 — equilibrium-augmented formulation: REAL training run

**Follow-up paper, Professor Rabczuk's items 2 and 3** (2026-10-16 email): "an equilibrium-augmented displacement formulation, adding equilibrium and traction residuals to the present energy loss," and "a more general H1/gradient or weak-form term that constrains displacement gradients more directly."

**A real technical finding worth stating plainly**: for a u-only network (no separate predicted stress field, unlike item 1's mixed-stress formulation), these two descriptions collapse to the SAME mechanism. B3's mesh uses trilinear (Hexa1) elements, whose shape-function gradients are constant within each element -- so a strong-form pointwise divergence residual on the stress implied by u is trivially zero within every element regardless of whether the field is physically correct (the standard reason finite-element practice uses the weak form, not a strong-form residual, for low-order elements). The one well-posed realization of "equilibrium residual" for a u-only network is therefore the weak form (virtual work): `integral P:grad(v) dV = 0` for admissible virtual fields v, which is simultaneously a gradient-constraining term in exactly the sense item 3 describes.

**Verified before any GPU time**: `test_equilibrium_augmented_B3.py` confirms the residual genuinely tracks equilibrium violation (it decreases as total potential energy is minimized, on a CPU toy model) -- not a disconnected number. A fixed-batch diagnostic at production mesh scale showed the loss stays stable (no divergence) with the default `w_equilibrium=1.0`; unlike item 1's mixed-stress formulation (which needed a ~28,000x reweighting after a severe imbalance was found), this term's magnitude is already roughly comparable to the energy term's own, so no drastic rescaling was needed -- though this was checked on a tiny toy model, same caveat as every other item here.

This checkpoint is still u-only (no new output channels), so the ORIGINAL `evaluate_B3_qois.py` applies unchanged for evaluation -- no new eval script was needed for this item.

In [ ]:
# =====================================================================
#  CELL -- B3 equilibrium-augmented: REAL training run (50,000
#  iterations, production model size, same budget as the DEM baseline),
#  then evaluation with the ORIGINAL evaluate_B3_qois.py (this
#  checkpoint is u-only, same output shape as the baseline).
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')

print(f'\n{"=" * 90}')
print('=== Re-running the equilibrium-residual correctness check in this environment ===')
from omar_pfem.test_equilibrium_augmented_B3 import test_residual_tracks_energy_minimization
test_residual_tracks_energy_minimization()

from omar_pfem.train_B3_equilibrium_augmented import get_eq_args, train
from omar_pfem.data.data_generate_B3_dataset import DEFAULT_RESOLUTION

device = torch.device('cuda')
R = '/content/drive/MyDrive/pfem_run'
OUTPUT_DIR = f'{R}/b3_equilibrium_augmented'
os.makedirs(OUTPUT_DIR, exist_ok=True)

args = get_eq_args([
    '--n_iters', '50000',
    '--batch_size', '8',
    '--log_every', '500',
    '--ckpt_every', '5000',
    '--output_dir', OUTPUT_DIR,
])
n_elem = (DEFAULT_RESOLUTION[0] - 1) * (DEFAULT_RESOLUTION[1] - 1) * (DEFAULT_RESOLUTION[2] - 1)
print(f'\nTraining at {DEFAULT_RESOLUTION} -> {n_elem} elements, {args.n_iters} iterations, '
      f'w_equilibrium={args.w_equilibrium}, n_virtual={args.n_virtual}, checkpoints -> {OUTPUT_DIR}')
print('Watch the printed energy vs. weak_form_residual magnitudes in the first few hundred '
      'iterations -- if one is wildly dominating the other (orders of magnitude apart, the '
      'same pattern mixed-stress hit before reweighting), interrupt and report the observed '
      'values rather than letting 50,000 iterations run on a possibly-unbalanced loss.')

torch.cuda.reset_peak_memory_stats(device)
t0 = time.time()
model = train(args, device)
elapsed_total = time.time() - t0

print(f'\n{"=" * 90}')
print(f'Training finished in {elapsed_total:.1f}s ({elapsed_total / args.n_iters:.4f}s/iteration average).')
print(f'GPU peak memory: {torch.cuda.max_memory_allocated(device) / 1e6:.1f}MB allocated.')

print(f'\n{"=" * 90}')
print('=== Evaluating the final checkpoint (original evaluate_B3_qois.py -- u-only model) ===')
CHECKPOINT = f'{OUTPUT_DIR}/checkpoint_{args.n_iters}.pt'
DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
OUT_JSON = f'{OUTPUT_DIR}/qois_{args.n_iters}.json'

sys.argv = [
    'evaluate_B3_qois.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--out_json', OUT_JSON,
]
from omar_pfem.evaluate_B3_qois import main as evaluate_main
evaluate_main()

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(OUTPUT_DIR, kind='b3_equilibrium_augmented_training',
                    args={'n_iters': args.n_iters, 'batch_size': args.batch_size,
                          'w_equilibrium': args.w_equilibrium, 'n_virtual': args.n_virtual},
                    started_at=_started,
                    results={'elapsed_total_s': elapsed_total,
                             's_per_iter': elapsed_total / args.n_iters},
                    outputs=[CHECKPOINT, OUT_JSON],
                    notes="First real training run of the equilibrium-augmented B3 "
                          "formulation (Professor Rabczuk's items 2/3, 2026-10-16 email) "
                          "-- u-only network, energy loss plus a weak-form virtual-work "
                          "equilibrium residual. Preceded by a correctness check "
                          "(residual tracks energy minimization) and a fixed-batch "
                          "diagnostic (stable, roughly balanced magnitudes) before any "
                          "real training time was spent.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back.')
